# Scratch TFLite Similarity Training

이 노트북은 `scratch` 점수 옵션에서 사용할 TCN/GCN TFLite 모델을 데이터 구성부터 학습, 저장, 런타임 확인까지 실행할 수 있도록 만든 end-to-end 노트북이다.

`scratch` 모델은 인터넷에서 받은 사전학습 모델이 아니라, 현재 프로젝트의 `data/reference_dances/*/reference.npy` skeleton 데이터로 직접 학습한다.

## 1. 환경과 프로젝트 루트 확인

In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'scripts' / 'train_scratch_similarity.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('pjt_main project root를 찾지 못했습니다.')

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.executable)

PROJECT_ROOT = /workspace/users/yijin/boot_env/pjt_main
Python = /workspace/users/yijin/boot_env/.venv/bin/python


## 2. Reference Skeleton 데이터 확인

학습 원본은 pose estimation이 끝난 `reference.npy` 파일이다. 일반적으로 shape는 `(frames, 33, 4)`이고 마지막 차원은 `x, y, z, visibility`이다.

In [2]:
import numpy as np

reference_paths = sorted((PROJECT_ROOT / 'data' / 'reference_dances').glob('*/reference.npy'))
for path in reference_paths:
    arr = np.load(path, allow_pickle=True)
    print(f'{path.parent.name:16s} shape={arr.shape} finite={np.isfinite(arr).mean():.6f}')

assert reference_paths, 'reference.npy 파일이 필요합니다.'

beginner_wave    shape=(1800, 33, 4) finite=1.000000
cheerup_dance    shape=(724, 33, 4) finite=1.000000
freestyle_free   shape=(3600, 33, 4) finite=1.000000
hiphop_move      shape=(792, 33, 4) finite=1.000000
kpop_basic       shape=(2700, 33, 4) finite=1.000000


## 3. 모델 입력 데이터 구성

모델은 전체 33개 landmark가 아니라 춤 비교에 중요한 12개 joint를 사용한다.

`DANCE_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]`

전처리 과정:

1. 12개 joint 선택
2. `feature_dims=2`이면 x,y만 사용
3. 골반 중심을 원점으로 이동
4. 어깨 너비로 scale 정규화
5. 최근 `sequence_length` 프레임을 쌓아 window 생성

In [3]:
from pose.landmark_utils import DANCE_JOINTS
from scoring.scratch_features import normalize_pose_landmarks, build_pose_window

SEQUENCE_LENGTH = 30
FEATURE_DIMS = 2

sample = np.load(reference_paths[0], allow_pickle=True)
one_frame = normalize_pose_landmarks(sample[0], DANCE_JOINTS, FEATURE_DIMS)
one_window = build_pose_window(sample, SEQUENCE_LENGTH - 1, SEQUENCE_LENGTH, DANCE_JOINTS, FEATURE_DIMS)

print('DANCE_JOINTS =', DANCE_JOINTS)
print('one_frame shape =', one_frame.shape)
print('one_window shape =', one_window.shape)
print('TFLite input shape =', (1, *one_window.shape))

DANCE_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
one_frame shape = (12, 2)
one_window shape = (30, 12, 2)
TFLite input shape = (1, 30, 12, 2)


## 4. Positive/Negative Pair 생성

별도 라벨 파일 없이 reference sequence에서 학습 pair를 만든다.

- Positive: 같은 춤에서 시간적으로 가까운 두 window, label `1`
- Negative: 다른 춤의 window 또는 같은 춤에서 충분히 멀리 떨어진 window, label `0`

기본값은 `positive_jitter=6`, `negative_gap=45`, `noise_std=0.015`이다.

In [4]:
from scripts.train_scratch_similarity import load_sequences, PairBatchGenerator

sequences = load_sequences('data/reference_dances', SEQUENCE_LENGTH)
gen = PairBatchGenerator(
    sequences=sequences,
    sequence_length=SEQUENCE_LENGTH,
    feature_dims=FEATURE_DIMS,
    batch_size=8,
    steps_per_epoch=1,
    positive_jitter=6,
    negative_gap=45,
    noise_std=0.015,
    seed=42,
)
(users, refs), labels = gen[0]
print('users =', users.shape)
print('refs =', refs.shape)
print('labels =', labels.tolist())
print('finite =', np.isfinite(users).all(), np.isfinite(refs).all())

users = (8, 30, 12, 2)
refs = (8, 30, 12, 2)
labels = [0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0]
finite = True True


## 5. 모델 구조 확인

`model_type=tcn`은 시간축 Conv1D 기반 모델이고, `model_type=gcn`은 skeleton adjacency 기반 graph aggregation과 temporal Conv를 함께 사용한다.

학습은 Siamese 구조로 한다. 두 window가 같은 encoder를 공유하고, 두 embedding의 cosine similarity가 label `0/1`에 맞도록 binary cross entropy로 학습된다. 서비스에는 Siamese 전체가 아니라 encoder 하나만 `.tflite`로 저장한다.

In [5]:
from scripts.train_scratch_similarity import import_tensorflow, build_encoder

tf = import_tensorflow()
for model_type in ['tcn', 'gcn']:
    encoder = build_encoder(
        tf,
        sequence_length=SEQUENCE_LENGTH,
        num_joints=len(DANCE_JOINTS),
        feature_dims=FEATURE_DIMS,
        embedding_dim=32,
        filters=32,
        blocks=2,
        kernel_size=3,
        dropout=0.1,
        model_type=model_type,
    )
    print(model_type, encoder.input_shape, encoder.output_shape, 'params=', encoder.count_params())

2026-04-15 21:49:43.492263: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 21:49:43.506679: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 21:49:43.506703: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 21:49:43.517459: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-15 21:49:44.103287: W tensorflow/compiler/tf

tcn (None, 30, 12, 2) (None, 32) params= 15840
gcn (None, 30, 12, 2) (None, 32) params= 78112


2026-04-15 21:49:45.138976: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-15 21:49:45.198338: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## 6. 모델 학습

`CONFIGS` 리스트에 학습하고 싶은 모델을 원하는 만큼 넣고, `RUN_TRAINING=True`로 바꾸면 리스트 순서대로 하나씩 학습한다.

- `SHARED` — 모든 config에 공통으로 적용되는 학습 루프/샘플링 기본값
- 각 config의 key는 `SHARED`를 덮어쓴다. 즉 `CONFIGS`에 쓴 항목만 변하고, 빼먹은 항목은 `SHARED` 값을 그대로 쓴다.

### `loss_type` 선택
- `'bce'` — 기본. Siamese + cosine에 대한 binary cross entropy. (user, ref, 0/1 label) 쌍으로 학습.
- `'triplet'` — margin 기반 triplet loss. (anchor, positive, negative) 3개 window를 받아 "positive가 negative보다 `triplet_margin`만큼 더 가깝도록" 학습.

`'triplet'`은 metric learning에서 분리가 더 잘 되는 경향이 있다. `triplet_margin`은 L2 정규화된 cosine 기반이라 0.1~0.5 사이 값을 쓴다.

In [10]:
# === 학습 설정 ===
RUN_TRAINING = True #False

# 모든 config에 공통으로 적용되는 기본값 (config에 같은 key를 넣으면 override됨)
SHARED = {
    'loss_type': 'bce',          # 'bce' 또는 'triplet'
    'triplet_margin': 0.2,       # triplet loss 전용 (bce일 땐 무시됨)
    'epochs': 30,
    'patience': 5,               # val_loss 개선 없을 때 조기 종료까지의 epoch 수
    'batch_size': 32,
    'steps_per_epoch': 60,
    'validation_steps': 10,
    'learning_rate': 1e-3,
    'positive_jitter': 6,
    'negative_gap': 600,
    'noise_std': 0.015,
    'seed': 42,
    'quantize': True,
}

# 학습할 모델 리스트. 원하는 만큼 추가/삭제
CONFIGS = [
    # BCE 
    {'name': 'my_gcn_e32_bce', 'model_type': 'gcn',
     'embedding_dim': 32, 'filters': 32, 'blocks': 2, 'kernel_size': 3, 'dropout': 0.10},
    {'name': 'my_gcn_e64_bce', 'model_type': 'gcn',
     'embedding_dim': 64, 'filters': 64, 'blocks': 3, 'kernel_size': 5, 'dropout': 0.15},
    # 같은 구조에 triplet loss만 교체 — 비교용
    {'name': 'my_gcn_e32_triplet', 'model_type': 'gcn',
     'embedding_dim': 32, 'filters': 32, 'blocks': 2, 'kernel_size': 3, 'dropout': 0.10,
     'loss_type': 'triplet', 'triplet_margin': 0.2},
    {'name': 'my_gcn_e64_triplet', 'model_type': 'gcn',
     'embedding_dim': 64, 'filters': 64, 'blocks': 3, 'kernel_size': 5, 'dropout': 0.15,
     'loss_type': 'triplet', 'triplet_margin': 0.2},
    # TCN + triplet
    {'name': 'my_tcn_e64_bce', 'model_type': 'tcn',
     'embedding_dim': 64, 'filters': 64, 'blocks': 4, 'kernel_size': 5, 'dropout': 0.15},
    {'name': 'my_tcn_e64_triplet', 'model_type': 'tcn',
     'embedding_dim': 64, 'filters': 64, 'blocks': 4, 'kernel_size': 5, 'dropout': 0.15,
     'loss_type': 'triplet', 'triplet_margin': 0.3, 'learning_rate': 8e-4},
    # TCN embedding 32 (경량) — bce / triplet 비교
    {'name': 'my_tcn_e32_bce', 'model_type': 'tcn',
     'embedding_dim': 32, 'filters': 32, 'blocks': 3, 'kernel_size': 3, 'dropout': 0.10},
    {'name': 'my_tcn_e32_triplet', 'model_type': 'tcn',
     'embedding_dim': 32, 'filters': 32, 'blocks': 3, 'kernel_size': 3, 'dropout': 0.10,
     'loss_type': 'triplet', 'triplet_margin': 0.2},
]


def build_cmd(cfg):
    merged = {**SHARED, **cfg}
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'train_scratch_similarity.py'),
        '--data-dir', 'data/reference_dances',
        '--model-name', merged['name'],
        '--model-type', merged['model_type'],
        '--sequence-length', str(SEQUENCE_LENGTH),
        '--feature-dims', str(FEATURE_DIMS),
        '--embedding-dim', str(merged['embedding_dim']),
        '--filters', str(merged['filters']),
        '--blocks', str(merged['blocks']),
        '--kernel-size', str(merged['kernel_size']),
        '--dropout', str(merged['dropout']),
        '--epochs', str(merged['epochs']),
        '--patience', str(merged['patience']),
        '--batch-size', str(merged['batch_size']),
        '--steps-per-epoch', str(merged['steps_per_epoch']),
        '--validation-steps', str(merged['validation_steps']),
        '--learning-rate', str(merged['learning_rate']),
        '--loss-type', merged['loss_type'],
        '--triplet-margin', str(merged['triplet_margin']),
        '--positive-jitter', str(merged['positive_jitter']),
        '--negative-gap', str(merged['negative_gap']),
        '--noise-std', str(merged['noise_std']),
        '--seed', str(merged['seed']),
    ]
    if not merged.get('quantize', True):
        cmd.append('--no-quantize')
    return cmd


for cfg in CONFIGS:
    cmd = build_cmd(cfg)
    print(f'\n[{cfg["name"]}] ' + ' '.join(cmd))
    if RUN_TRAINING:
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

if not RUN_TRAINING:
    print(f'\n{len(CONFIGS)}개 모델 준비 완료. RUN_TRAINING=True로 바꾸면 순차 학습됩니다.')


[my_gcn_e32_bce] /workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/train_scratch_similarity.py --data-dir data/reference_dances --model-name my_gcn_e32_bce --model-type gcn --sequence-length 30 --feature-dims 2 --embedding-dim 32 --filters 32 --blocks 2 --kernel-size 3 --dropout 0.1 --epochs 30 --patience 5 --batch-size 32 --steps-per-epoch 60 --validation-steps 10 --learning-rate 0.001 --loss-type bce --triplet-margin 0.2 --positive-jitter 6 --negative-gap 600 --noise-std 0.015 --seed 42
[DATA] loaded 5 reference sequences
  - beginner_wave: 1800 frames (33, 4)
  - hiphop_move: 792 frames (33, 4)
  - kpop_basic: 2700 frames (33, 4)
  - cheerup_dance: 724 frames (33, 4)
  - freestyle_free: 3600 frames (33, 4)
[DATA] model input window shape: (30, 12, 2)
[MODEL] name=my_gcn_e32_bce type=gcn


2026-04-15 21:56:51.946572: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 21:56:51.960602: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 21:56:51.960624: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 21:56:52.556324: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-04-15 21:56:53.901023: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the gu

Model: "scratch_siamese_similarity"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_window         │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼��───────────┼───────────────────┤
│ reference_window    │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scratch_gcn_encoder │ (None, 32)        │     78,112 │ user_window[0][0… │
│ (Functional)        │                   │            │ reference_window… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity   │ (None, 1)      

2026-04-15 21:59:10.922820: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
W0000 00:00:1776257951.112077 1301611 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776257951.112104 1301611 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


[SAVE] TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/scratch/my_gcn_e32_bce.tflite (86.8 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=1.6701
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/scratch/my_gcn_e32_bce_meta.json

Run with:
  python src/main.py -s scratch --scratch-model-name my_gcn_e32_bce

[my_gcn_e64_bce] /workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/train_scratch_similarity.py --data-dir data/reference_dances --model-name my_gcn_e64_bce --model-type gcn --sequence-length 30 --feature-dims 2 --embedding-dim 64 --filters 64 --blocks 3 --kernel-size 5 --dropout 0.15 --epochs 30 --patience 5 --batch-size 32 --steps-per-epoch 60 --validation-steps 10 --learning-rate 0.001 --loss-type bce --triplet-margin 0.2 --positive-jitter 6 --negative-gap 600 --noise-std 0.015 --seed 42
[DATA] loaded 5 reference sequences
  - beginner_wave: 1800 frames (33, 4)
  - hiphop_move

...

[SAVE] TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/scratch/my_tcn_e32_bce.tflite (37.3 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=3.5409
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/scratch/my_tcn_e32_bce_meta.json

Run with:
  python src/main.py -s scratch --scratch-model-name my_tcn_e32_bce

[my_tcn_e32_triplet] /workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/train_scratch_similarity.py --data-dir data/reference_dances --model-name my_tcn_e32_triplet --model-type tcn --sequence-length 30 --feature-dims 2 --embedding-dim 32 --filters 32 --blocks 3 --kernel-size 3 --dropout 0.1 --epochs 30 --patience 5 --batch-size 32 --steps-per-epoch 60 --validation-steps 10 --learning-rate 0.001 --loss-type triplet --triplet-margin 0.2 --positive-jitter 6 --negative-gap 600 --noise-std 0.015 --seed 42
[DATA] loaded 5 reference sequences
  - beginner_wave: 1800 frames (33, 4)
  - 

2026-04-15 22:10:06.004320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 22:10:06.018643: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 22:10:06.018665: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 22:10:06.602949: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-04-15 22:10:07.944922: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the gu

Model: "scratch_triplet_similarity"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_window       │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼��───────────┼───────────────────┤
│ positive_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼────────────────���──┤
│ scratch_tcn_encoder │ (None, 32)   

2026-04-15 22:11:57.849574: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
W0000 00:00:1776258718.055394 1314762 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776258718.055419 1314762 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


[SAVE] TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/scratch/my_tcn_e32_triplet.tflite (37.3 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=2.0259
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/scratch/my_tcn_e32_triplet_meta.json

Run with:
  python src/main.py -s scratch --scratch-model-name my_tcn_e32_triplet


## 7. 학습 결과 확인

`CONFIGS`에 들어 있는 모든 모델에 대해 `data/models/scratch/{name}_meta.json`을 읽어 학습 config와 loss/metric history를 요약한다.

각 학습마다 아래 세 파일이 만들어진다.

- `{name}.tflite` — 서비스용 런타임 모델
- `{name}_encoder.keras` — 재변환/재학습용 Keras 모델
- `{name}_meta.json` — config + history

**History 키는 loss_type에 따라 다르다**:

- `bce` → `loss`, `val_loss`, `binary_accuracy`, `val_binary_accuracy`
- `triplet` → `loss`, `val_loss`, `positive_margin`, `val_positive_margin`, `violation_rate`, `val_violation_rate`
  - `positive_margin` = 평균적으로 positive가 negative보다 얼마나 더 가까운가 (**높을수록 좋음**)
  - `violation_rate` = triplet 제약을 아직 위반하는 샘플 비율 (**낮을수록 좋음**, 0이면 모든 anchor가 positive를 negative보다 margin 이상 가깝게 학습함)

In [11]:
models_dir = PROJECT_ROOT / 'data' / 'models' / 'scratch'

for cfg in CONFIGS:
    meta_path = models_dir / f'{cfg["name"]}_meta.json'
    print('='*60)
    print(f'[{cfg["name"]}] {meta_path.name}')
    if not meta_path.exists():
        print('  (아직 학습되지 않음)')
        continue
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    config = meta.get('config') or {}
    print(f'  created_at = {meta.get("created_at")}')
    print(f'  model      = {meta.get("model_name")} ({meta.get("model_type")})')
    print(f'  loss_type  = {config.get("loss_type", "bce")}', end='')
    if config.get('loss_type') == 'triplet':
        print(f'  (margin={config.get("triplet_margin")})')
    else:
        print()
    print(f'  input/out  = {meta.get("input_shape")} -> {meta.get("output_shape")}')
    history = meta.get('history') or {}
    # epochs actually run (best epoch은 그 이전일 수 있음 — restore_best_weights로 복원됨)
    any_vals = next(iter(history.values()), [])
    if any_vals:
        print(f'  epochs_run = {len(any_vals)}')
    for key, values in history.items():
        if values:
            print(f'  {key:22s} last={values[-1]:.4f}')

[my_gcn_e32_bce] my_gcn_e32_bce_meta.json
  created_at = 2026-04-15 21:59:11
  model      = my_gcn_e32_bce (gcn)
  loss_type  = bce
  input/out  = [1, 30, 12, 2] -> [1, 32]
  epochs_run = 30
  binary_accuracy        last=0.8104
  loss                   last=0.3660
  val_binary_accuracy    last=0.8000
  val_loss               last=0.3481
[my_gcn_e64_bce] my_gcn_e64_bce_meta.json
  created_at = 2026-04-15 22:00:21
  model      = my_gcn_e64_bce (gcn)
  loss_type  = bce
  input/out  = [1, 30, 12, 2] -> [1, 64]
  epochs_run = 10
  binary_accuracy        last=0.8036
  loss                   last=0.3971
  val_binary_accuracy    last=0.7844
  val_loss               last=0.3747
[my_gcn_e32_triplet] my_gcn_e32_triplet_meta.json
  created_at = 2026-04-15 22:02:46
  model      = my_gcn_e32_triplet (gcn)
  loss_type  = triplet  (margin=0.2)
  input/out  = [1, 30, 12, 2] -> [1, 32]
  epochs_run = 21
  loss                   last=0.0053
  positive_margin        last=0.7792
  val_loss               la

In [15]:

# ============================================================
# 실제 reference 데이터를 애니메이션으로 띄우고, 학습한 encoder가
# 각 window 쌍에 부여하는 cosine similarity를 확인한다.
#
# 시각화: 전처리 전 raw MediaPipe 좌표 (0~1 range, 12 joints)
# 임베딩: 전처리된 window (골반 중심 정규화, 어깨 너비 스케일)
# ============================================================
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- 설정: 보여줄 모델과 비교할 window들 ---
VIS_MODEL_NAME = 'my_gcn_e64_triplet'   # CONFIGS 중 학습이 끝난 모델명

# (라벨, 춤 이름, end_frame)  end_frame=None이면 임의 선택
SAMPLES = [
    ('anchor',   'cheerup_dance', 300),   # 기준
    ('positive', 'cheerup_dance', 308),   # 같은 춤, 근처 → cos ↑ 기대
    ('negative', 'hiphop_move',   300),   # 다른 춤        → cos ↓ 기대
]

# --- 12개 joint를 잇는 skeleton edge (DANCE_JOINTS의 로컬 index) ---
# 0:L어깨 1:R어깨 2:L팔꿈치 3:R팔꿈치 4:L손목 5:R손목
# 6:L골반 7:R골반 8:L무릎 9:R무릎 10:L발목 11:R발목
DANCE_EDGES = [
    (0, 1), (6, 7),               # 어깨선, 골반선
    (0, 6), (1, 7),               # 몸통 옆선
    (0, 2), (2, 4),               # 왼팔
    (1, 3), (3, 5),               # 오른팔
    (6, 8), (8, 10),              # 왼다리
    (7, 9), (9, 11),              # 오른다리
]

COLOR_BY_LABEL = {
    'anchor':    'tab:gray',
    'reference': 'tab:gray',
    'positive':  'tab:green',
    'negative':  'tab:red',
}
DEFAULT_COLOR = 'tab:blue'

# --- 1) 각 샘플의 window 뽑기 ---
#   raw_window  : 전처리 전 MediaPipe 원본 xy (시각화용)  shape (T, 12, 2)
#   window      : 전처리된 normalized xy (임베딩 입력용)  shape (T, 12, 2)
reference_dir = PROJECT_ROOT / 'data' / 'reference_dances'
rng = np.random.default_rng(0)

entries = []
for label, dance_name, end_frame in SAMPLES:
    seq = np.load(reference_dir / dance_name / 'reference.npy').astype(np.float32)
    if end_frame is None:
        end_frame = int(rng.integers(SEQUENCE_LENGTH - 1, len(seq)))
    end_frame = max(SEQUENCE_LENGTH - 1, min(int(end_frame), len(seq) - 1))

    # 전처리된 window (모델 입력용)
    window = build_pose_window(
        seq, end_frame, SEQUENCE_LENGTH,
        target_joints=DANCE_JOINTS, feature_dims=FEATURE_DIMS,
    )

    # raw window: joint 선택만 하고 정규화 없음 (시각화용)
    start_frame = end_frame - SEQUENCE_LENGTH + 1
    raw_window = seq[start_frame:end_frame + 1][:, DANCE_JOINTS, :2].copy()  # (T, 12, 2)
    # MediaPipe x,y 는 [0,1] 범위, NaN 제거
    raw_window = np.nan_to_num(raw_window, nan=0.0)

    entries.append({'label': label, 'dance': dance_name,
                    'end': end_frame, 'window': window, 'raw_window': raw_window})

# --- 2) 학습된 TFLite encoder 로드 ---
tflite_path = models_dir / f'{VIS_MODEL_NAME}.tflite'
meta_path   = models_dir / f'{VIS_MODEL_NAME}_meta.json'
assert tflite_path.exists(), f'{tflite_path.name} 가 없음 — 먼저 학습하세요.'
meta = json.loads(meta_path.read_text('utf-8'))

interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
in_det  = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]


def embed(window):
    """(T, J, C) → (D,) embedding. 양자화 여부도 투명 처리."""
    tensor = window[None, ...].astype(np.float32)
    if not np.issubdtype(in_det['dtype'], np.floating):
        scale, zero = in_det.get('quantization', (0.0, 0))
        if scale:
            q = np.round(tensor / scale + zero)
            info = np.iinfo(in_det['dtype'])
            tensor = np.clip(q, info.min, info.max).astype(in_det['dtype'])
    interp.set_tensor(in_det['index'], tensor)
    interp.invoke()
    out = interp.get_tensor(out_det['index'])
    if not np.issubdtype(out.dtype, np.floating):
        scale, zero = out_det.get('quantization', (0.0, 0))
        if scale:
            out = (out.astype(np.float32) - zero) * scale
    return out.reshape(-1).astype(np.float32)


def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu < 1e-8 or nv < 1e-8:
        return 0.0
    return float(np.dot(u, v) / (nu * nv))


# --- 3) pairwise cosine 행렬 출력 (임베딩은 전처리된 window 사용) ---
for e in entries:
    e['emb'] = embed(e['window'])

labels = [e['label'] for e in entries]
print(f'[MODEL] {VIS_MODEL_NAME}  '
      f'(loss_type={meta["config"].get("loss_type","bce")}, '
      f'embedding_dim={meta["config"].get("embedding_dim")})')
print()
print('pairwise cosine similarity:')
header = ' ' * 12 + ''.join(f'{lb:>11s}' for lb in labels)
print(header)
for i, ei in enumerate(entries):
    row = f'{ei["label"]:>10s}: '
    for j, ej in enumerate(entries):
        row += f'{cosine(ei["emb"], ej["emb"]):+10.4f} '
    print(row)

# --- 4) 애니메이션: raw 좌표 기준 시각화 ---
def axis_limits_raw(raw_windows, pad=0.03):
    """raw MediaPipe 좌표 (0~1) 기준 axis 범위."""
    arr = np.stack(raw_windows, axis=0)  # (N, T, J, 2)
    x, y = arr[..., 0], arr[..., 1]
    # 유효값(0 이외)만 고려
    valid_x = x[x > 0.01]
    valid_y = y[y > 0.01]
    if len(valid_x) == 0:
        return (0.0, 1.0), (0.0, 1.0)
    return (
        (float(valid_x.min()) - pad, float(valid_x.max()) + pad),
        (float(valid_y.min()) - pad, float(valid_y.max()) + pad),
    )


def draw_skeleton_raw(ax, pts, color):
    """pts: (J, 2) raw MediaPipe xy. y축 image 관례로 flip."""
    xs, ys = pts[:, 0], -pts[:, 1]
    scatter = ax.scatter(xs, ys, c=color, s=30, zorder=3)
    lines = []
    for a, b in DANCE_EDGES:
        line = ax.plot([xs[a], xs[b]], [ys[a], ys[b]],
                       color=color, lw=1.8, alpha=0.85)[0]
        lines.append(line)
    return scatter, lines


xlim, (y_lo, y_hi) = axis_limits_raw([e['raw_window'] for e in entries])
ylim = (-y_hi, -y_lo)   # y flip 반영
anchor_emb = entries[0]['emb']

fig, axes = plt.subplots(1, len(entries), figsize=(4 * len(entries), 4.5))
if len(entries) == 1:
    axes = [axes]

artists_by_ax = []
for ax, e in zip(axes, entries):
    sim = cosine(anchor_emb, e['emb'])
    ax.set_title(
        f"{e['label']}\n{e['dance']}  (end={e['end']})\n"
        f"cos(anchor, .) = {sim:+.3f}",
        fontsize=10,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    artists_by_ax.append([])

frame_text = fig.text(0.5, 0.02, '', ha='center')


def update(f):
    for old in artists_by_ax:
        for a in old:
            a.remove()
        old.clear()
    for ax, e, old in zip(axes, entries, artists_by_ax):
        color = COLOR_BY_LABEL.get(e['label'], DEFAULT_COLOR)
        # 시각화: raw_window 사용
        scatter, lines = draw_skeleton_raw(ax, e['raw_window'][f], color)
        old.extend([scatter, *lines])
    frame_text.set_text(f'frame {f + 1} / {SEQUENCE_LENGTH}')
    arts = [frame_text]
    for old in artists_by_ax:
        arts.extend(old)
    return arts


anim = FuncAnimation(
    fig, update, frames=SEQUENCE_LENGTH,
    interval=1000 / 30, blit=False,
)
plt.close(fig)
HTML(anim.to_jshtml())


[MODEL] my_gcn_e64_triplet  (loss_type=triplet, embedding_dim=64)

pairwise cosine similarity:
                 anchor   positive   negative
    anchor:    +1.0000    +0.9987    -0.0594 
  positive:    +0.9987    +1.0000    -0.0684 
  negative:    -0.0594    -0.0684    +1.0000 


## 8. Runtime Smoke Test

학습된 모델이 있다면 서비스와 같은 `ScratchPoseSimilarity.compute()` 경로로 점수가 finite하게 나오는지 확인한다.

In [9]:
from scoring.scratch_similarity import ScratchPoseSimilarity, resolve_scratch_model_path


def smoke_test(model_name, sequence_length, feature_dims):
    model_path = resolve_scratch_model_path(model_name, str(models_dir))
    ref = np.load(PROJECT_ROOT / 'data' / 'reference_dances' / 'beginner_wave' / 'reference.npy')
    other = np.load(PROJECT_ROOT / 'data' / 'reference_dances' / 'hiphop_move' / 'reference.npy')
    comparator = ScratchPoseSimilarity(
        model_path, sequence_length=sequence_length, feature_dims=feature_dims)
    same = None
    for idx in range(sequence_length):
        same = comparator.compute(ref[idx], ref, idx)
    comparator.reset()
    cross = None
    for idx in range(sequence_length):
        cross = comparator.compute(other[idx], ref, idx)
    print(f'{model_name:20s} same={same}  cross={cross}  '
          f'finite={np.isfinite(same) and np.isfinite(cross)}')


for cfg in CONFIGS:
    tflite_path = models_dir / f'{cfg["name"]}.tflite'
    if not tflite_path.exists():
        print(f'{cfg["name"]:20s} (아직 학습되지 않음)')
        continue
    smoke_test(cfg['name'], SEQUENCE_LENGTH, FEATURE_DIMS)

my_gcn_e32           same=1.0  cross=0.13904376327991486  finite=True
my_gcn_e64           same=1.0  cross=0.0018449326744303107  finite=True
my_tcn_e64           same=1.0  cross=0.5316677093505859  finite=True


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 9. 서비스 실행

`CONFIGS`에 넣은 이름 중 하나를 `--scratch-model-name`으로 지정한다.

```bash
python src/main.py -s scratch --scratch-model-name my_gcn_e64
```

직접 경로를 넣고 싶으면:

```bash
python src/main.py -s scratch --scratch-model-path data/models/scratch/my_gcn_e64.tflite
```

`SEQUENCE_LENGTH`/`FEATURE_DIMS`를 기본값(30, 2)에서 바꿔 학습했다면 실행 시에도 같이 전달해야 한다.

```bash
python src/main.py -s scratch \
    --scratch-model-name my_gcn_e64 \
    --scratch-sequence-length 30 \
    --scratch-feature-dims 2
```